# Potentiostat discovery notebook

Barebones notebook for discovering USB-connected potentiostats on Linux.

Goals:
- do not use the archived potentiostat package
- do not use workflow adapters
- only inspect what serial devices are visible
- try opening each candidate device directly
- collect enough information to build a clean modular driver later

## Notes

This notebook is intentionally minimal. It only uses:
- Python standard library
- optional `pyserial` if installed

It does not import anything from the archived potentiostat code.

In [ ]:
from pathlib import Path
import glob
import os
import platform
import subprocess
import sys
import time

try:
    import serial
    import serial.tools.list_ports
    HAS_PYSERIAL = True
except Exception as exc:
    HAS_PYSERIAL = False
    SERIAL_IMPORT_ERROR = exc

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'pyserial available: {HAS_PYSERIAL}')
if not HAS_PYSERIAL:
    print(f'pyserial import error: {SERIAL_IMPORT_ERROR}')

In [ ]:
def list_serial_candidates():
    candidates = sorted(set(glob.glob('/dev/ttyACM*') + glob.glob('/dev/ttyUSB*')))
    return candidates

candidates = list_serial_candidates()
print('Candidate serial devices:')
for path in candidates:
    print(' -', path)

if not candidates:
    print('No /dev/ttyACM* or /dev/ttyUSB* devices found.')

In [ ]:
if HAS_PYSERIAL:
    print('Detailed port information from pyserial:')
    ports = list(serial.tools.list_ports.comports())
    for p in ports:
        print('-' * 60)
        print(f'device      : {p.device}')
        print(f'name        : {p.name}')
        print(f'description : {p.description}')
        print(f'hwid        : {p.hwid}')
        print(f'vid         : {p.vid}')
        print(f'pid         : {p.pid}')
        print(f'manufacturer: {p.manufacturer}')
        print(f'product     : {p.product}')
        print(f'serial no   : {p.serial_number}')
        print(f'location    : {p.location}')
    if not ports:
        print('pyserial sees no ports.')
else:
    print('Skipping detailed port inspection because pyserial is unavailable.')

In [ ]:
def read_udev_info(device_path: str):
    try:
        result = subprocess.run(
            ['udevadm', 'info', '--query=all', '--name', device_path],
            capture_output=True,
            text=True,
            check=False,
        )
        return result.stdout.strip()
    except FileNotFoundError:
        return 'udevadm not available'

for path in candidates:
    print('=' * 80)
    print(path)
    print(read_udev_info(path)[:4000])

In [ ]:
def try_open_port(device_path: str, baudrate: int = 115200, timeout: float = 1.0, read_seconds: float = 2.0):
    if not HAS_PYSERIAL:
        return {'device': device_path, 'opened': False, 'error': 'pyserial not installed'}

    result = {
        'device': device_path,
        'opened': False,
        'baudrate': baudrate,
        'bytes_read': 0,
        'preview_ascii': '',
        'preview_hex': '',
        'error': None,
    }

    try:
        with serial.Serial(device_path, baudrate=baudrate, timeout=timeout) as ser:
            result['opened'] = True
            ser.reset_input_buffer()
            time.sleep(read_seconds)
            waiting = ser.in_waiting
            payload = ser.read(waiting or 256)
            result['bytes_read'] = len(payload)
            result['preview_ascii'] = payload.decode('utf-8', errors='replace')[:500]
            result['preview_hex'] = payload[:64].hex(' ')
    except Exception as exc:
        result['error'] = repr(exc)

    return result

probe_results = [try_open_port(path) for path in candidates]
for item in probe_results:
    print('-' * 80)
    for key, value in item.items():
        print(f'{key}: {value}')

## Interpreting results

Typical outcomes:
- device path exists but no bytes are emitted: the board may wait for commands
- readable ASCII lines: the board is likely using a text serial protocol
- binary-looking bytes: the board may use a binary command protocol
- permission errors: add your user to `dialout` or adjust udev permissions

If four potentiostats are attached through a USB hub, this notebook should show four serial devices if the OS enumerates them separately.

In [ ]:
# Optional: choose one port manually for later low-level experiments
SELECTED_PORT = candidates[0] if candidates else None
SELECTED_PORT

In [ ]:
# Barebones low-level serial open test.
# This cell does not send any commands yet.

if SELECTED_PORT and HAS_PYSERIAL:
    with serial.Serial(SELECTED_PORT, baudrate=115200, timeout=1) as ser:
        print('Opened:', ser.port)
        print('Baudrate:', ser.baudrate)
        print('Timeout:', ser.timeout)
        print('in_waiting:', ser.in_waiting)
else:
    print('No selected port or pyserial unavailable.')

## Recommended next step

Once you know which ports belong to the potentiostats, the next clean step is to build a new module in this folder with a small interface such as:
- `list_devices()`
- `connect(port)`
- `read_identity()`
- `read_raw()`
- `write_raw()`

Then the measurement protocol can be layered on top after discovery is confirmed.

## Adapter-driven tests

This section uses the repo adapter directly (`src/adapters/potentiostat_adapter.py`) instead of ad-hoc serial code.

What this verifies:
- adapter import and initialization
- port discovery and open/close checks
- firmware ping (`READ_SWITCH`, `READ_GAIN`)
- minimal measurement probe path used by workflow nodes (`sdl1ElectrochemicalMeasurement`)


In [1]:
from pathlib import Path
import sys
import json

repo_root = Path.cwd()
while repo_root.name != 'AC-OTFlex-monorepo' and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.adapters.potentiostat_adapter import PotentiostatAdapter

adapter_cfg = {
    'ports': [f'/dev/ttyACM{i}' for i in range(4)],
    'baudrate': 115200,
    'timeout_s': 2.0,
    'device_id': 1,
    'strict': False,
}

adapter = PotentiostatAdapter(adapter_cfg, root_dir=repo_root)
print('Adapter ready from:', repo_root)


Adapter ready from: /mnt/storage/Projects/ACSDL1/AC-OTFlex-monorepo


In [2]:
smoke = adapter.smoke_test()
print(json.dumps(smoke, indent=2))

serial_rows = adapter.list_serial_devices()
print(f'\npyserial device rows: {len(serial_rows)}')
for row in serial_rows:
    print('-', row.get('device'), '|', row.get('description'), '|', row.get('serial_number'))


{
  "expected_ports": [
    "/dev/ttyACM0",
    "/dev/ttyACM1",
    "/dev/ttyACM2",
    "/dev/ttyACM3"
  ],
  "detected_acm": [
    "/dev/ttyACM0",
    "/dev/ttyACM1",
    "/dev/ttyACM2",
    "/dev/ttyACM3"
  ],
  "detected_usb": [],
  "missing": [],
  "open_results": [
    {
      "port": "/dev/ttyACM0",
      "opened": true,
      "error": null
    },
    {
      "port": "/dev/ttyACM1",
      "opened": true,
      "error": null
    },
    {
      "port": "/dev/ttyACM2",
      "opened": true,
      "error": null
    },
    {
      "port": "/dev/ttyACM3",
      "opened": true,
      "error": null
    }
  ]
}

pyserial device rows: 36
- /dev/ttyS31 | n/a | None
- /dev/ttyS30 | n/a | None
- /dev/ttyS29 | n/a | None
- /dev/ttyS28 | n/a | None
- /dev/ttyS27 | n/a | None
- /dev/ttyS26 | n/a | None
- /dev/ttyS25 | n/a | None
- /dev/ttyS24 | n/a | None
- /dev/ttyS23 | n/a | None
- /dev/ttyS22 | n/a | None
- /dev/ttyS21 | n/a | None
- /dev/ttyS20 | n/a | None
- /dev/ttyS19 | n/a | None
- /dev/

In [3]:
await adapter.connect()

probe_params = {
    'uo_name': 'Notebook_Adapter_Probe',
    'measurement_type': 'ADAPTER_PROBE',
    'com_port': '/dev/ttyACM0',
    'channel': 1,
    'sequence_enabled': False,
    'potentiostat_configs': [
        {'com_port': '/dev/ttyACM0', 'row': 'A', 'file_name': 'A_probe'},
        {'com_port': '/dev/ttyACM1', 'row': 'B', 'file_name': 'B_probe'},
        {'com_port': '/dev/ttyACM2', 'row': 'C', 'file_name': 'C_probe'},
        {'com_port': '/dev/ttyACM3', 'row': 'D', 'file_name': 'D_probe'},
    ],
}

probe_result = await adapter.echem_measure(probe_params)
print(json.dumps(probe_result, indent=2))

await adapter.disconnect()


[Potentiostat] Ready (baud=115200, timeout=2.0s, device_id=1, expected_ports=['/dev/ttyACM0', '/dev/ttyACM1', '/dev/ttyACM2', '/dev/ttyACM3'])
[Potentiostat] Notebook_Adapter_Probe: probing 4 port(s)
  - /dev/ttyACM0: OK; open_err=None ping_err=None
  - /dev/ttyACM1: OK; open_err=None ping_err=None
  - /dev/ttyACM2: OK; open_err=None ping_err=None
  - /dev/ttyACM3: OK; open_err=None ping_err=None
{
  "ok": true,
  "ports": [
    "/dev/ttyACM0",
    "/dev/ttyACM1",
    "/dev/ttyACM2",
    "/dev/ttyACM3"
  ],
  "checks": [
    {
      "port": "/dev/ttyACM0",
      "open": {
        "port": "/dev/ttyACM0",
        "opened": true,
        "error": null
      },
      "ping": {
        "port": "/dev/ttyACM0",
        "alive": true,
        "switch_on": false,
        "gain": 4,
        "gain_label": "1 M\u03a9",
        "error": null
      },
      "adc": {
        "port": "/dev/ttyACM0",
        "readings": {
          "VREF": {
            "raw_V": 0.0331,
            "scaled_V": -4.8998
